# PKG payment regularity: analysis before the frequency metrics

**PNC Treasury Management · DSI · Payment Knowledge Graph**

How regular are the payments in the graph, and how often does a regular payment arrive late, get skipped or stop?
The answers decide the frequency metrics of customer metrics v4 and whether the contagion question (does a customer
change behaviour when a regular incoming payment is delayed or stops, and does it differ for daily, weekly or monthly
payments) has enough events to study. This notebook only reads; it writes scratch files, no tables.

**Stream.** All payments from one payer to one payee over the window, on any rail, one point per payment day (amounts
of the same day are added). External counterparties are taken at `ent_l2` of `pkg_cpty_entity_map_v2`, so a company
paying from a new account or bank stays the same stream. Flow types:

| flow | payer → payee |
|---|---|
| `in` | external counterparty → PNC customer |
| `out` | PNC customer → external counterparty |
| `book` | PNC customer → another PNC customer (book to book) |
| `self` | PNC customer → the same customer (own accounts) |

**Definitions** (all thresholds are settings; §3 shows how the results move with them)
- **Gap:** days between consecutive payment days of a stream. **Cadence** from the median gap: daily (business days),
  weekly, biweekly / semimonthly, monthly, bimonthly, quarterly; anything else is *irregular interval*. Business days
  skip weekends and US Federal Reserve holidays (a holiday on a Sunday moves to Monday).
- **On time:** a gap within `max(TOL_MIN, TOL_FRAC × median gap)` days of the median gap (daily: the next business
  day). **Regular stream:** at least `MIN_PAYMENTS` payment days, a known cadence and at least `REG_MIN` of its gaps on
  time. **Recurring, irregular:** enough payments but not regular.
- **Late:** a gap longer than on time but shorter than `SKIP_MULT ×` median; **skipped:** a gap of at least
  `SKIP_MULT ×` median (a payment missed, then the stream continued); **stopped:** a regular stream with no payment for
  at least `SKIP_MULT ×` its median gap at the end of the window.
- **Amount class** (streams with enough payments): *fixed* (at least `FIXED_SHARE` of payments within `FIXED_PCT` of
  the stream's median amount), *stepped* (not fixed, at most `STEP_MAX` amount changes, e.g. a raise), *bounded*
  (amount CV at most `BOUNDED_CV`), *variable* (the rest).

**Sample.** `SAMPLE_PCT` of customers (by a hash of `mdm_id`) with all of their streams, so customer-level results are
complete for every sampled customer. Shares are reliable at 10%; dollar shares move more because of very large streams,
so the final run can use 100.

**Sections:** §1 stream days · §2 streams · §3 how much is regular · §4 amounts · §5 timing · §6 late, skipped and
stopped · §7 customers who depend on regular income · §8 figures (PNG files in `FIG_DIR`) · §9 housekeeping.

**Run order:** after `pkg_edges_refresh.ipynb` and `pkg_reference_data.ipynb`. §1 and §2 build scratch files and are
reused on a rerun with the same settings; §3–§7 only read them.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
SHUFFLE_PARTITIONS = 2000

# ---- Window and sample (a change rebuilds §1 and §2)
WINDOW_END    = None      # last month analysed, e.g. "2026-10"; None = latest month published by pkg_edges_refresh
WINDOW_MONTHS = 12        # months in the window; quarterly streams need at least 12
SAMPLE_PCT    = 10        # % of customers (hash of mdm_id) whose streams are analysed; 100 = every customer

# ---- Timing (a change rebuilds §2)
TOL_FRAC, TOL_MIN = 0.15, 1   # on time: |gap − median gap| ≤ max(TOL_MIN, TOL_FRAC × median gap) days
SKIP_MULT  = 1.75             # a gap ≥ SKIP_MULT × median gap = a missed payment
FIXED_PCT  = 0.01             # "same amount": within ±1% of the stream's median amount
CHANGE_PCT = 0.01             # an amount change between consecutive payments larger than this counts as a change

# ---- Classification (applied when §3–§7 read the streams; no rebuild)
MIN_PAYMENTS = 4          # payment days a stream needs before its regularity is judged
REG_MIN      = 0.70       # regular: at least this share of gaps on time (one late payment costs two gaps)
FIXED_SHARE  = 0.80       # fixed amount: at least this share of payments within FIXED_PCT of the median amount
STEP_MAX     = 2          # stepped: not fixed and at most this many amount changes
BOUNDED_CV   = 0.25       # bounded: coefficient of variation of the amount at most this
CADENCES = [              # (name, lowest, highest median gap in calendar days); daily = median business-day gap ≤ 1
    ("weekly", 6, 8), ("biweekly / semimonthly", 12, 17), ("monthly", 26, 35),
    ("bimonthly", 55, 66), ("quarterly", 84, 98)]

MASS_STOP_MIN = 10        # §6d: a payer that stops paying at least this many sampled customers in one month = payer-wide stop

# ---- Figures and scratch
FIG_DIR             = "figures/pkg_payment_regularity"   # PNG files, next to the notebook (one subfolder per window end)
KEEP_SCRATCH        = True    # keep §1/§2 scratch so §3–§7 can be rerun without rebuilding
CLEAN_OTHER_SCRATCH = False   # §8: delete scratch folders of other settings

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, hashlib, json, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_payment_regularity")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write; netty /
# TransportResponseHandler print harmless errors when executors are released. Their loggers are switched off.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    out_, m = [], a
    while m <= b:
        out_.append(m); m = month_add(m, 1)
    return out_

def us_bank_holidays(y0, y1):
    """US Federal Reserve holidays (ACH / Fedwire closed). A holiday on a Sunday is observed on Monday; one on a
    Saturday is not observed (the Fed stays open on Friday)."""
    def nth(y, m, wd, k):                      # k-th weekday wd (Mon = 0) of the month; k = -1 → last
        if k > 0:
            d = dt.date(y, m, 1)
            d += dt.timedelta(days=(wd - d.weekday()) % 7 + 7 * (k - 1))
        else:
            d = dt.date(y, m, calendar.monthrange(y, m)[1])
            d -= dt.timedelta(days=(d.weekday() - wd) % 7)
        return d
    out_ = []
    for y in range(y0, y1 + 1):
        fixed = [dt.date(y, 1, 1), dt.date(y, 6, 19), dt.date(y, 7, 4), dt.date(y, 11, 11), dt.date(y, 12, 25)]
        moving = [nth(y, 1, 0, 3), nth(y, 2, 0, 3), nth(y, 5, 0, -1), nth(y, 9, 0, 1), nth(y, 10, 0, 2), nth(y, 11, 3, 4)]
        for d in fixed:
            if d.weekday() == 6:
                d += dt.timedelta(days=1)
            if d.weekday() < 5:
                out_.append(d)
        out_ += moving
    return sorted(out_)
HOLIDAYS = us_bank_holidays(2023, 2028)

def bidx(c):
    """Business-day index of a date column: each business day (Monday–Friday, not a Fed holiday) counts 1; weekends
    and holidays share the next business day's index. So Friday → Monday is 1 business day, Friday → Tuesday after a
    Monday holiday is also 1, and a payment made on a weekend or holiday is on time for a daily stream."""
    n = F.datediff(c, F.to_date(F.lit("1970-01-05")))           # 1970-01-05 is a Monday
    wk = F.floor(n / 7)
    hol = sum(F.when(c > F.to_date(F.lit(h.isoformat())), 1).otherwise(0) for h in HOLIDAYS)
    return (wk * 5 + F.least(n - wk * 7, F.lit(5)) - hol).cast("long")

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

## §1 Stream days

Reads `pkg_edge_daily` for the window (every month must be published by `pkg_edges_refresh.ipynb`), keeps the sampled
customers' payments, maps external ids to `ent_l2`, and writes one row per stream and payment day with the gap to the
previous payment day (calendar days and business days, weekends and Fed holidays skipped).

In [ ]:
# ---- §1 Stream days -----------------------------------------------------------------------------
T_EDGE_D, T_MAP, T_CUST = f"{DB}.pkg_edge_daily", f"{DB}.pkg_cpty_entity_map_v2", f"{DB}.pkg_cust_dim"
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")

PROBLEMS = []
NEED = {T_EDGE_D: ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "amount", "n_txn",
                   "month"],
        T_MAP: ["cpty_id", "ent_l2", "name_class"],
        T_CUST: ["mdm_id", "party_type", "is_tm"]}
for t, cols in NEED.items():
    have = set(spark.table(t).columns)
    PROBLEMS += [{"problem": "column missing", "detail": f"{t}.{c}"} for c in cols if c not in have]
_loc = table_location(T_EDGE_D)
PARTS = sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {T_EDGE_D}").collect())
PUBLISHED = [m for m in PARTS if exists(f"{_loc}/month={m}/_PKG_BUILD")]
END = WINDOW_END or (PUBLISHED[-1] if PUBLISHED else None)
MONTHS = month_range(month_add(END, -(WINDOW_MONTHS - 1)), END) if END else []
_unpub = [m for m in MONTHS if m not in PUBLISHED]
if _unpub:
    PROBLEMS.append({"problem": "months not published by pkg_edges_refresh", "detail": ", ".join(_unpub)})
if PROBLEMS:
    out(sec(f"§1 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§1: {len(PROBLEMS)} problem(s), see the table above")

W_START = dt.date.fromisoformat(MONTHS[0] + "-01")
W_END = dt.date(int(END[:4]), int(END[5:7]), calendar.monthrange(int(END[:4]), int(END[5:7]))[1])
VERSION = "reg_v2"          # v2: business days skip Fed holidays
SIG_SD = hashlib.md5(json.dumps({"v": VERSION, "m": MONTHS, "s": SAMPLE_PCT}).encode()).hexdigest()[:10]
SIG_ST = hashlib.md5(json.dumps({"sd": SIG_SD, "t": [TOL_FRAC, TOL_MIN, SKIP_MULT, FIXED_PCT, CHANGE_PCT]})
                     .encode()).hexdigest()[:10]
SCR = f"{DB_LOC}/_tmp_pkg_regularity"
P_SD, P_ST = f"{SCR}/sd_{SIG_SD}", f"{SCR}/st_{SIG_ST}"
RAIL = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P", "7": "CHECK", "9": "PCARD"}
K = ["payer", "payer_kind", "payee", "payee_kind", "flow"]

def sampled(c):
    return F.lit(True) if SAMPLE_PCT >= 100 else F.expr(f"pmod(xxhash64({c}), 100) < {int(SAMPLE_PCT)}")

sd_built = not exists(f"{P_SD}/_SUCCESS")
if sd_built:
    with timed("§1 stream days"):
        src_c, dst_c = F.col("src_kind") == "CUST", F.col("dst_kind") == "CUST"
        e = (spark.table(T_EDGE_D).where(F.col("month").isin(MONTHS))
             .withColumn("payer_s", src_c & sampled("src_id"))
             .withColumn("payee_s", dst_c & sampled("dst_id"))
             .where(F.col("payer_s") | F.col("payee_s"))
             .withColumn("flow", F.when(src_c & dst_c & F.coalesce(F.col("is_self"), F.lit(False)), "self")
                                  .when(src_c & dst_c, "book").when(dst_c, "in").when(src_c, "out").otherwise("other"))
             .withColumn("rail", F.create_map(*[F.lit(x) for kv in RAIL.items() for x in kv])[F.substring("category", 1, 1)])
             .withColumn("cp_id", F.when(~src_c, F.col("src_id")).when(~dst_c, F.col("dst_id")))
             .select("txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "flow", "rail", "cp_id", "payer_s", "payee_s",
                     F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn")))
        mp = spark.table(T_MAP).select(F.col("cpty_id").alias("cp_id"), F.col("ent_l2").alias("cp_ent"),
                                       F.col("name_class").alias("cp_class"))
        # internal rows have no counterparty id: they skip the map join (NULL keys would all land in one task)
        e = (e.where(F.col("cp_id").isNotNull()).join(mp, "cp_id", "left")
              .unionByName(e.where(F.col("cp_id").isNull())
                            .withColumn("cp_ent", F.lit(None).cast("string"))
                            .withColumn("cp_class", F.lit(None).cast("string"))))
        e = (e.withColumn("payer", F.when(F.col("src_kind") == "CPTY", F.coalesce("cp_ent", "src_id")).otherwise(F.col("src_id")))
              .withColumn("payee", F.when(F.col("dst_kind") == "CPTY", F.coalesce("cp_ent", "dst_id")).otherwise(F.col("dst_id")))
              .withColumnRenamed("src_kind", "payer_kind").withColumnRenamed("dst_kind", "payee_kind"))
        sd = e.groupBy(*K, "txn_date").agg(
            F.sum("amount").alias("amt"), F.sum("n_txn").alias("n_txn"),
            F.max(F.struct("amount", "rail")).getField("rail").alias("rail"),
            F.max(F.struct("amount", "cp_class")).getField("cp_class").alias("cp_class"),
            F.max("payer_s").alias("payer_s"), F.max("payee_s").alias("payee_s"))
        w = Window.partitionBy("payer", "payee").orderBy("txn_date")
        sd = (sd.withColumn("prev_dt", F.lag("txn_date").over(w))
                .withColumn("prev_amt", F.lag("amt").over(w))
                .withColumn("gap", F.datediff("txn_date", "prev_dt"))
                .withColumn("gap_bd", bidx(F.col("txn_date")) - bidx(F.col("prev_dt"))))
        sd.write.mode("overwrite").parquet(P_SD)

SD = spark.read.parquet(P_SD)
s1 = (SD.groupBy("flow").agg(F.count("*").alias("payment_days"), F.sum("n_txn").alias("transactions"),
                             (F.sum("amt") / 1e9).alias("dollars_B"))
        .toPandas().sort_values("dollars_B", ascending=False))
_S1_OK = True
out(sec(f"§1 Stream days {MONTHS[0]} → {MONTHS[-1]}, {SAMPLE_PCT}% of customers — "
        + (f"built ({TIMES['§1 stream days']:.1f} min)" if sd_built else "reused"), tbl(s1, dec=("dollars_B",))),
    para(f"Window {W_START} → {W_END}. Scratch {P_SD}."))

## §2 Streams

One row per stream: payments, dollars, first and last payment, median gap, gaps on time / early / late / skipped,
lateness in days, amount spread around the median, amount changes, the most common weekday and day of month (±2
days), payer
and payee type (customer: `party_type` O = org, P = person; counterparty: name class business → org, person, bank,
other), and whether the customer side is TM.

In [ ]:
# ---- §2 Streams ---------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

DEV_CUTS = [("0_1pct", 0.001), ("1pct", 0.01), ("2pct", 0.02), ("5pct", 0.05), ("10pct", 0.10), ("25pct", 0.25),
            ("50pct", 0.50), ("100pct", 1.00)]
LAG_BINS = [("≤ -4", None, -4), ("-3..-1", -3, -1), ("0", 0, 0), ("+1..+2", 1, 2), ("+3..+5", 3, 5),
            ("+6..+10", 6, 10), ("+11..+20", 11, 20), ("> +20", 21, None)]
st_built = not exists(f"{P_ST}/_SUCCESS")
if st_built:
    with timed("§2 streams"):
        sd = spark.read.parquet(P_SD)
        s1 = sd.groupBy(*K).agg(
            F.count("*").alias("n_pay"), F.sum("n_txn").alias("n_txn"), F.sum("amt").alias("amt"),
            F.min("txn_date").alias("first_dt"), F.max("txn_date").alias("last_dt"),
            F.percentile_approx("gap", 0.5, 1000).alias("med_gap"),
            F.percentile_approx("gap_bd", 0.5, 1000).alias("med_gap_bd"),
            F.percentile_approx("amt", 0.5, 1000).alias("med_amt"),
            F.avg("amt").alias("mean_amt"), F.stddev_pop("amt").alias("sd_amt"),
            F.sum((F.col("n_txn") > 1).cast("long")).alias("multi_days"),
            F.max(F.struct("amt", "rail")).getField("rail").alias("rail"),
            F.max(F.struct("amt", "cp_class")).getField("cp_class").alias("cp_class"),
            F.max("payer_s").alias("payer_s"), F.max("payee_s").alias("payee_s"))
        s1.write.mode("overwrite").parquet(f"{P_ST}_s1")
        s1 = spark.read.parquet(f"{P_ST}_s1")

        j = sd.join(s1.select(*K, "med_gap", "med_gap_bd", "med_amt"), K)
        has = F.col("gap").isNotNull()
        daily = F.col("med_gap_bd") <= 1
        tol = F.greatest(F.lit(float(TOL_MIN)), F.col("med_gap") * float(TOL_FRAC))
        ddays = F.when(daily, F.col("gap_bd") - 1).otherwise(F.col("gap") - F.col("med_gap"))
        ontime = F.when(daily, F.col("gap_bd") <= 1).otherwise(F.abs(F.col("gap") - F.col("med_gap")) <= tol)
        skip = F.when(daily, F.col("gap_bd") >= 2).otherwise(F.col("gap") >= float(SKIP_MULT) * F.col("med_gap"))
        j = (j.withColumn("ontime", has & ontime).withColumn("skip", has & skip)
              .withColumn("late", has & ~ontime & ~skip & (ddays > 0))
              .withColumn("early", has & ~ontime & (ddays < 0))
              .withColumn("ddays", F.when(has, ddays))
              .withColumn("dev", F.when(F.col("med_amt") > 0, F.abs(F.col("amt") - F.col("med_amt")) / F.col("med_amt")))
              .withColumn("chg_size", F.when(F.col("prev_amt") > 0,
                                             F.abs(F.col("amt") - F.col("prev_amt")) / F.col("prev_amt")))
              .withColumn("chg", F.coalesce(F.col("chg_size") > float(CHANGE_PCT), F.lit(False))))
        def cnt(c):
            return F.sum(F.coalesce(c, F.lit(False)).cast("long"))
        def lag_cond(lo, hi):
            c = F.col("ddays").isNotNull()
            if lo is not None: c = c & (F.col("ddays") >= lo)
            if hi is not None: c = c & (F.col("ddays") <= hi)
            return c
        a = j.groupBy(*K).agg(
            cnt(has).alias("n_gaps"), cnt(F.col("ontime")).alias("n_ontime"), cnt(F.col("late")).alias("n_late"),
            cnt(F.col("early")).alias("n_early"), cnt(F.col("skip")).alias("n_skip"),
            *[cnt(lag_cond(lo, hi)).alias(f"lag_{i}") for i, (_, lo, hi) in enumerate(LAG_BINS)],
            cnt(F.col("dev") <= float(FIXED_PCT)).alias("w_fixed"),
            *[cnt(F.col("dev") <= x).alias(f"w_{n}") for n, x in DEV_CUTS],
            cnt(F.col("chg")).alias("n_changes"),
            F.avg(F.when(F.col("chg"), F.col("chg_size"))).alias("chg_size"))
        dw = (sd.groupBy(*K, F.dayofweek("txn_date").alias("d")).count()
                .groupBy(*K).agg(F.max("count").alias("top_dow_n")))
        # day of month counted within ±2 days (a payment due on the 1st moves to the 3rd after a weekend)
        dm = (sd.groupBy(*K, F.dayofmonth("txn_date").alias("d")).count()
                .withColumn("near", F.sum("count").over(Window.partitionBy(*K).orderBy("d").rangeBetween(-2, 2)))
                .groupBy(*K).agg(F.max("near").alias("top_dom_n")))
        cu = spark.table(T_CUST).select("mdm_id", F.upper(F.trim("party_type")).alias("pt"),
                                        F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("tm"))
        cu = cu.dropDuplicates(["mdm_id"])
        def ctype(kind, pt, cls):
            return (F.when(F.col(kind) == "CUST",
                           F.when(F.col(pt) == "O", "org").when(F.col(pt) == "P", "person").otherwise("unknown"))
                     .otherwise(F.when(F.col(cls) == "business", "org").when(F.col(cls) == "person", "person")
                                 .when(F.col(cls) == "bank", "bank").otherwise("other")))
        st = (s1.join(a, K).join(dw, K).join(dm, K)
                .join(cu.select(F.col("mdm_id").alias("payer"), F.col("pt").alias("payer_pt"), F.col("tm").alias("payer_tm")),
                      "payer", "left")
                .join(cu.select(F.col("mdm_id").alias("payee"), F.col("pt").alias("payee_pt"), F.col("tm").alias("payee_tm")),
                      "payee", "left")
                .withColumn("payer_type", ctype("payer_kind", "payer_pt", "cp_class"))
                .withColumn("payee_type", ctype("payee_kind", "payee_pt", "cp_class"))
                .withColumn("cust_tm", F.when(F.col("flow") == "in", F.coalesce("payee_tm", F.lit(False)))
                                        .when(F.col("flow") == "out", F.coalesce("payer_tm", F.lit(False)))
                                        .otherwise(F.coalesce("payer_tm", F.lit(False)) | F.coalesce("payee_tm", F.lit(False))))
                .drop("payer_pt", "payee_pt"))
        st.write.mode("overwrite").parquet(P_ST)
        delete(f"{P_ST}_s1")

# ---- classification (settings applied here, no rebuild)
CAD_ORDER = ["daily"] + [c[0] for c in CADENCES] + ["irregular interval", "few payments", "one-off"]
REGULAR_CADS = ["daily"] + [c[0] for c in CADENCES]
def classify(st):
    cad_known = F.lit(None).cast("string")
    for name, lo, hi in reversed(CADENCES):
        cad_known = F.when(F.col("med_gap").between(lo, hi), F.lit(name)).otherwise(cad_known)
    cad_known = F.when(F.col("med_gap_bd") <= 1, F.lit("daily")).otherwise(cad_known)
    st = (st.withColumn("cad_known", F.when(F.col("n_pay") >= 2, cad_known))
            .withColumn("ontime_share", F.col("n_ontime") / F.col("n_gaps")))
    cad = (F.when(F.col("n_pay") == 1, "one-off").when(F.col("n_pay") < MIN_PAYMENTS, "few payments")
            .otherwise(F.coalesce(F.col("cad_known"), F.lit("irregular interval"))))
    reg = (F.col("n_pay") >= MIN_PAYMENTS) & F.col("cad_known").isNotNull() & (F.col("ontime_share") >= REG_MIN)
    amt_class = (F.when(F.col("n_pay") < MIN_PAYMENTS, F.lit(None))
                  .when(F.col("w_fixed") / F.col("n_pay") >= FIXED_SHARE, "fixed")
                  .when(F.col("n_changes") <= STEP_MAX, "stepped")
                  .when(F.col("sd_amt") / F.col("mean_amt") <= BOUNDED_CV, "bounded")
                  .otherwise("variable"))
    w_end = F.to_date(F.lit(W_END.isoformat()))
    idle = F.datediff(w_end, F.col("last_dt"))
    idle_bd = bidx(w_end) - bidx(F.col("last_dt"))
    stopped = F.coalesce(reg & F.when(F.col("cad_known") == "daily", idle_bd >= 2)
                                .otherwise(idle >= float(SKIP_MULT) * F.col("med_gap")), F.lit(False))
    expected = F.expr("date_add(last_dt, cast(med_gap as int))")
    exp_end = F.when(stopped, F.least(expected, w_end)).otherwise(w_end)
    return (st.withColumn("cadence", cad).withColumn("regular", F.coalesce(reg, F.lit(False)))
              .withColumn("group", F.when(F.col("regular"), "regular")
                                    .when(F.col("n_pay") >= MIN_PAYMENTS, "recurring, irregular")
                                    .when(F.col("n_pay") > 1, "few payments").otherwise("one-off"))
              .withColumn("amount_class", amt_class).withColumn("cv", F.col("sd_amt") / F.col("mean_amt"))
              .withColumn("stopped", stopped).withColumn("expected_dt", F.when(stopped, expected))
              .withColumn("exposure_m", F.greatest(F.datediff(exp_end, F.col("first_dt")), F.lit(0)) / 30.44)
              .withColumn("top_dow_share", F.col("top_dow_n") / F.col("n_pay"))
              .withColumn("top_dom_share", F.col("top_dom_n") / F.col("n_pay")))

ST = classify(spark.read.parquet(P_ST)).cache()
n_st = ST.count()
FLOWS = ["in", "out", "book", "self"]
s2 = (ST.groupBy("flow").agg(F.count("*").alias("streams"), F.sum("n_pay").alias("payment_days"),
                             (F.sum("amt") / 1e9).alias("dollars_B"))
        .toPandas())
s2["flow"] = pd.Categorical(s2.flow, FLOWS + ["other"]); s2 = s2.sort_values("flow")
out(sec(f"§2 Streams: {n_st:,} — " + (f"built ({TIMES['§2 streams']:.1f} min)" if st_built else "reused"),
        tbl(s2, dec=("dollars_B",))),
    para(f"Scratch {P_ST}. Classification: MIN_PAYMENTS {MIN_PAYMENTS}, REG_MIN {REG_MIN}, tolerance "
         f"max({TOL_MIN}, {TOL_FRAC} × median gap) days, skipped ≥ {SKIP_MULT} × median."))

## §3 How much is regular

By flow: share of streams and of dollars that are regular, recurring but irregular, a few payments, or one-off; the
cadence mix of regular streams; payer → payee type (org → person into a customer looks like payroll); rails; TM
customers; and how the regular share moves with `REG_MIN` and `MIN_PAYMENTS`.

In [ ]:
# ---- §3 How much is regular ---------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

GROUPS = ["regular", "recurring, irregular", "few payments", "one-off"]
def by_group(df, keys):
    g = df.groupBy(*keys, "group").agg(F.count("*").alias("n"), F.sum("amt").alias("a")).toPandas()
    n = g.pivot_table(index=keys, columns="group", values="n", aggfunc="sum", fill_value=0)
    a = g.pivot_table(index=keys, columns="group", values="a", aggfunc="sum", fill_value=0)
    res = pd.DataFrame({"streams": n.sum(axis=1).astype("int64"), "dollars_B": a.sum(axis=1) / 1e9})
    for c in GROUPS:
        res[f"{c} (streams)"] = (n[c] if c in n else 0) / res.streams
    for c in GROUPS[:2]:
        res[f"{c} (dollars)"] = (a[c] if c in a else 0) / (res.dollars_B * 1e9)
    return res.reset_index()
PCT_G = tuple([f"{c} (streams)" for c in GROUPS] + [f"{c} (dollars)" for c in GROUPS[:2]])
def order_flows(df):
    df["flow"] = pd.Categorical(df.flow, FLOWS + ["other"]); return df.sort_values(list(df.columns[:1]))

t_flow = order_flows(by_group(ST, ["flow"]))
t_tm = order_flows(by_group(ST.where("cust_tm"), ["flow"]))

reg = ST.where("regular")
c = reg.groupBy("flow", "cadence").agg(F.count("*").alias("n"), F.sum("amt").alias("a")).toPandas()
c["n_sh"] = c.n / c.groupby("flow").n.transform("sum"); c["a_sh"] = c.a / c.groupby("flow").a.transform("sum")
t_cad = pd.DataFrame({"cadence": [x for x in CAD_ORDER if x in set(c.cadence)]})
for fl in FLOWS:
    x = c[c.flow == fl].set_index("cadence")
    t_cad[f"{fl} streams"] = t_cad.cadence.map(x.n_sh).fillna(0)
    t_cad[f"{fl} dollars"] = t_cad.cadence.map(x.a_sh).fillna(0)

pt = (ST.groupBy("flow", "payer_type", "payee_type")
        .agg(F.count("*").alias("streams"), F.sum(F.col("regular").cast("long")).alias("regular_streams"),
             (F.sum("amt") / 1e9).alias("dollars_B"), F.sum(F.when(F.col("regular"), F.col("amt"))).alias("ra"))
        .toPandas())
pt["regular (streams)"] = pt.regular_streams / pt.streams
pt["regular (dollars)"] = pt.ra.fillna(0) / (pt.dollars_B * 1e9)
pt["share of flow streams"] = pt.streams / pt.groupby("flow").streams.transform("sum")
pt = order_flows(pt).sort_values(["flow", "streams"], ascending=[True, False])
pt = pt[pt["share of flow streams"] >= 0.005]

rl = by_group(ST.where(F.col("flow").isin("in", "out")), ["rail"]).sort_values("streams", ascending=False)

# sensitivity: regular share at other REG_MIN / MIN_PAYMENTS values (cadence known, enough payments, share on time)
sens_rows = []
tot = ST.agg(F.count("*").alias("n"), F.sum("amt").alias("a")).first()
aggs = []
for mp_ in (3, 4, 6):
    for r_ in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
        cond = (F.col("n_pay") >= mp_) & F.col("cad_known").isNotNull() & (F.col("ontime_share") >= r_)
        aggs += [F.sum(cond.cast("long")).alias(f"n_{mp_}_{r_}"), F.sum(F.when(cond, F.col("amt"))).alias(f"a_{mp_}_{r_}")]
sv = ST.agg(*aggs).first().asDict()
for mp_ in (3, 4, 6):
    row = {"MIN_PAYMENTS": mp_}
    for r_ in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
        row[f"REG_MIN {r_} streams"] = sv[f"n_{mp_}_{r_}"] / tot["n"]
        row[f"REG_MIN {r_} dollars"] = (sv[f"a_{mp_}_{r_}"] or 0) / tot["a"]
    sens_rows.append(row)
sens = pd.DataFrame(sens_rows)

out(sec("§3a By flow (all customers in the sample)", tbl(t_flow, pct=PCT_G, dec=("dollars_B",))),
    sec("§3b By flow, TM customers only (customer side is TM)", tbl(t_tm, pct=PCT_G, dec=("dollars_B",))),
    sec("§3c Cadence of regular streams: share of each flow's regular streams and regular dollars",
        tbl(t_cad, pct=tuple(t_cad.columns[1:]))),
    sec("§3d Payer → payee type (rows with at least 0.5% of the flow's streams)",
        tbl(pt[["flow", "payer_type", "payee_type", "streams", "share of flow streams", "regular (streams)",
                "regular (dollars)", "dollars_B"]],
            pct=("share of flow streams", "regular (streams)", "regular (dollars)"), dec=("dollars_B",))),
    sec("§3e By rail (in + out)", tbl(rl, pct=PCT_G, dec=("dollars_B",))),
    sec("§3f Sensitivity: share of all streams and dollars that are regular", tbl(sens, pct=tuple(sens.columns[1:]))),
    para("TOL_FRAC changes need a rebuild of §2; MIN_PAYMENTS and REG_MIN apply instantly."))

## §4 Amounts

Is the amount of a regular payment the same every time? Amount class by cadence and flow; how close payments sit to
the stream's median amount (pooled over payments); for stepped streams, the size of the change (a raise or an
inflation adjustment shows as one change of a few percent); the spread (CV) of streams that are not fixed; and how
often a payment day holds more than one transaction (its amount is then a daily total).

In [ ]:
# ---- §4 Amounts ---------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

AC = ["fixed", "stepped", "bounded", "variable"]
def class_mix(keys):
    g = reg.groupBy(*keys, "amount_class").agg(F.count("*").alias("n"), F.sum("amt").alias("a")).toPandas()
    n = g.pivot_table(index=keys, columns="amount_class", values="n", aggfunc="sum", fill_value=0)
    a = g.pivot_table(index=keys, columns="amount_class", values="a", aggfunc="sum", fill_value=0)
    res = pd.DataFrame({"regular streams": n.sum(axis=1).astype("int64")})
    for x in AC:
        res[f"{x} (streams)"] = (n[x] if x in n else 0) / res["regular streams"]
    for x in AC:
        res[f"{x} (dollars)"] = (a[x] if x in a else 0) / a.sum(axis=1)
    return res.reset_index()
PCT_AC = tuple([f"{x} (streams)" for x in AC] + [f"{x} (dollars)" for x in AC])
def order_cad(df):
    df["cadence"] = pd.Categorical(df.cadence, CAD_ORDER); return df.sort_values("cadence")
m_cad = order_cad(class_mix(["cadence"]))
m_flow = order_flows(class_mix(["flow"]))

DEV_NAMES = ["0_1pct", "1pct", "2pct", "5pct", "10pct", "25pct", "50pct", "100pct"]
DEV_LABELS = ["±0.1%", "±1%", "±2%", "±5%", "±10%", "±25%", "±50%", "±100%"]
def dev_table(df, keys):
    g = df.groupBy(*keys).agg(F.sum("n_pay").alias("payments"),
                              *[F.sum(f"w_{n}").alias(n) for n in DEV_NAMES]).toPandas()
    for n, l in zip(DEV_NAMES, DEV_LABELS):
        g[l] = g[n] / g.payments
    return g[keys + ["payments"] + DEV_LABELS]
d_cad = order_cad(dev_table(reg, ["cadence"]))
d_grp = dev_table(ST.where(F.col("n_pay") >= MIN_PAYMENTS), ["group"])

stp = (reg.where(F.col("amount_class") == "stepped").where(F.col("n_changes") > 0)
          .withColumn("size", F.when(F.col("chg_size") <= 0.02, "1–2%").when(F.col("chg_size") <= 0.05, "2–5%")
                               .when(F.col("chg_size") <= 0.10, "5–10%").when(F.col("chg_size") <= 0.25, "10–25%")
                               .when(F.col("chg_size") <= 0.50, "25–50%").otherwise("> 50%"))
          .groupBy("cadence", "size").count().toPandas())
SIZES = ["1–2%", "2–5%", "5–10%", "10–25%", "25–50%", "> 50%"]
st_t = stp.pivot_table(index="cadence", columns="size", values="count", aggfunc="sum", fill_value=0)
st_t = st_t.reindex(columns=[s for s in SIZES if s in st_t.columns])
st_t = (st_t.div(st_t.sum(axis=1), axis=0).assign(streams=st_t.sum(axis=1).astype("int64"))
            .reset_index())
st_t = order_cad(st_t)[["cadence", "streams"] + [s for s in SIZES if s in st_t.columns]]

cvq = (reg.where(F.col("amount_class") != "fixed").groupBy("cadence")
          .agg(F.count("*").alias("streams"), F.percentile_approx("cv", [0.25, 0.5, 0.75, 0.9], 1000).alias("q"))
          .toPandas())
for i, l in enumerate(["CV p25", "CV p50", "CV p75", "CV p90"]):
    cvq[l] = cvq.q.map(lambda v: float(v[i]) if v is not None else None)
cvq = order_cad(cvq.drop(columns="q"))
md_ = (ST.where(F.col("n_pay") >= MIN_PAYMENTS).groupBy("group")
         .agg((F.sum("multi_days") / F.sum("n_pay")).alias("payment days with 2+ transactions")).toPandas())

out(sec("§4a Amount class of regular streams by cadence", tbl(m_cad, pct=PCT_AC)),
    sec("§4b Amount class of regular streams by flow", tbl(m_flow, pct=PCT_AC)),
    sec("§4c Share of payments within ±x of the stream's median amount (regular streams, by cadence)",
        tbl(d_cad, pct=tuple(DEV_LABELS))),
    sec("§4d Same, regular vs recurring-irregular streams", tbl(d_grp, pct=tuple(DEV_LABELS))),
    sec("§4e Stepped regular streams: average size of an amount change", tbl(st_t, pct=tuple(SIZES))),
    sec("§4f Spread (CV) of regular streams that are not fixed", tbl(cvq, dec=("CV p25", "CV p50", "CV p75", "CV p90"))),
    sec("§4g Payment days with more than one transaction", tbl(md_, pct=("payment days with 2+ transactions",))),
    para(f"fixed: ≥ {FIXED_SHARE:.0%} of payments within ±{FIXED_PCT:.0%} of the median; stepped: ≤ {STEP_MAX} changes "
         f"> {CHANGE_PCT:.0%}; bounded: CV ≤ {BOUNDED_CV}."))

## §5 Timing

For regular streams: share of gaps on time, early, late and skipped; how many days a payment arrives after (or before)
the stream's usual gap; and whether payments land on a fixed weekday (weekly, biweekly) or a fixed day of the month
within ±2 days (monthly and longer) — a fixed calendar day makes the expected date of the next payment easy to set.

In [ ]:
# ---- §5 Timing ----------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

tg = (reg.groupBy("cadence").agg(F.sum("n_gaps").alias("gaps"), F.sum("n_ontime").alias("on"),
                                 F.sum("n_early").alias("ea"), F.sum("n_late").alias("la"), F.sum("n_skip").alias("sk"),
                                 *[F.sum(f"lag_{i}").alias(f"lag_{i}") for i in range(len(LAG_BINS))])
         .toPandas())
t5a = pd.DataFrame({"cadence": tg.cadence, "gaps": tg.gaps, "on time": tg.on / tg.gaps, "early": tg.ea / tg.gaps,
                    "late": tg.la / tg.gaps, "skipped": tg.sk / tg.gaps})
t5b = pd.DataFrame({"cadence": tg.cadence})
for i, (lab, _, _) in enumerate(LAG_BINS):
    t5b[lab] = tg[f"lag_{i}"] / tg.gaps
t5a, t5b = order_cad(t5a), order_cad(t5b)

anc = (reg.withColumn("share", F.when(F.col("cadence").isin("weekly", "biweekly / semimonthly"),
                                      F.col("top_dow_share"))
                               .when(F.col("cadence").isin("monthly", "bimonthly", "quarterly"), F.col("top_dom_share")))
          .where(F.col("share").isNotNull())
          .withColumn("anchor", F.when(F.col("share") >= 0.8, "≥ 80% on one day")
                                 .when(F.col("share") >= 0.5, "50–80%").otherwise("< 50%"))
          .groupBy("cadence", "anchor").count().toPandas())
an = anc.pivot_table(index="cadence", columns="anchor", values="count", aggfunc="sum", fill_value=0)
an = an.div(an.sum(axis=1), axis=0).reset_index()
an = order_cad(an)[["cadence"] + [c for c in ["≥ 80% on one day", "50–80%", "< 50%"] if c in an.columns]]

out(sec("§5a Gaps of regular streams", tbl(t5a, pct=("on time", "early", "late", "skipped"))),
    sec("§5b Days after the usual gap (negative = early; daily: business days after the next business day)",
        tbl(t5b, pct=tuple(l for l, _, _ in LAG_BINS))),
    sec("§5c Fixed calendar day: weekday (weekly, biweekly) or day of month ±2 days (monthly and longer)",
        tbl(an, pct=tuple(an.columns[1:]))),
    para("A late payment usually costs two gaps (one long, the next short), which is why REG_MIN is below 0.8."))

## §6 Late, skipped and stopped

How often a regular stream misses its date, by cadence and flow, and how many regular streams stop. A stop is dated
by its expected payment date (last payment + median gap); stops in the last months of the window are under-counted
for long cadences, because a quarterly stream needs about five months of silence to count. Stop rate is per 100
stream-months of exposure. For stopped incoming streams (`in` and `book`, sampled receiving customers), the last two
tables show whether the receiving customer was still active after the expected date — the cases where a behaviour
change could be observed.

In [ ]:
# ---- §6 Late, skipped and stopped --------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

b = (reg.groupBy("flow", "cadence")
        .agg(F.count("*").alias("regular streams"), F.avg((F.col("n_late") > 0).cast("double")).alias("≥ 1 late"),
             F.avg((F.col("n_skip") > 0).cast("double")).alias("≥ 1 skipped"),
             F.avg(F.col("stopped").cast("double")).alias("stopped"),
             F.sum(F.col("stopped").cast("long")).alias("stops"), F.sum("exposure_m").alias("exp_m"))
        .toPandas())
b["stops per 100 stream-months"] = 100 * b.stops / b.exp_m
b = order_flows(order_cad(b)).sort_values(["flow", "cadence"]).drop(columns=["exp_m"])

stops = reg.where("stopped")
sm = (stops.withColumn("m", F.date_format("expected_dt", "yyyy-MM"))
           .withColumn("side", F.when(F.col("flow").isin("in", "book"), "incoming (in + book)").otherwise(F.col("flow")))
           .groupBy("m", "side").count().toPandas()
           .pivot_table(index="m", columns="side", values="count", aggfunc="sum", fill_value=0).reset_index()
           .rename(columns={"m": "expected month"}))

# receiving customers' activity after the stop (sampled receivers only: all their streams are in the sample)
cust_pay = (ST.where((F.col("payer_kind") == "CUST") & F.col("payer_s"))
              .groupBy(F.col("payer").alias("cust")).agg(F.max("last_dt").alias("last_out")))
cust_rcv = (ST.where((F.col("payee_kind") == "CUST") & F.col("payee_s"))
              .groupBy(F.col("payee").alias("cust")).agg(F.max("last_dt").alias("last_in")))
act = cust_pay.join(cust_rcv, "cust", "full")
inc_stops = (stops.where(F.col("flow").isin("in", "book") & F.col("payee_s"))
                  .select(F.col("payee").alias("cust"), "cadence", "expected_dt", "amount_class", "cust_tm")
                  .join(act, "cust", "left")
                  .withColumn("pays_after", F.coalesce(F.col("last_out") > F.col("expected_dt"), F.lit(False)))
                  .withColumn("receives_after", F.coalesce(F.col("last_in") > F.col("expected_dt"), F.lit(False))))
ia = (inc_stops.groupBy("cadence")
               .agg(F.count("*").alias("stopped incoming streams"),
                    F.countDistinct("cust").alias("receiving customers"),
                    F.avg(F.col("pays_after").cast("double")).alias("customer still pays out after"),
                    F.avg(F.col("receives_after").cast("double")).alias("customer still receives other money after"),
                    F.avg(F.col("cust_tm").cast("double")).alias("TM customer"))
               .toPandas())
ia = order_cad(ia)
PCT_IA = ("customer still pays out after", "customer still receives other money after", "TM customer")
out(sec("§6a Regular streams that were late, skipped or stopped", tbl(b, pct=("≥ 1 late", "≥ 1 skipped", "stopped"),
                                                                       dec=("stops per 100 stream-months",))),
    sec("§6b Stops by expected payment month (last months under-counted)", tbl(sm)),
    sec("§6c Stopped regular incoming streams and the receiving customer afterwards", tbl(ia, pct=PCT_IA)),
    para("A stop can also be a payer changing its account at the same bank (people are not merged across ids) or "
         "a relationship moving to another PNC customer; §6c counts only what the edges show."))

### §6d Stops month by month: rate, payer-wide stops, largest payers

Checks whether a month with many stops (August in the first run) is a real event or a data change. For regular
incoming streams of sampled customers: stops and streams at risk by expected month (a stream is at risk from its
first payment to its stop or the window end), the stop rate by cadence, and the share of stops that are **payer-wide**
(the payer stopped paying at least `MASS_STOP_MIN` sampled customers in the same month: an employer closing, a program
ending, a feed change). Then the payers with the most stopped streams in the peak month, next to their usual month.
Payer-wide stops matter for the contagion study: they hit customers from outside, so the customer did not choose them.

In [ ]:
# ---- §6d Stops month by month -------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

inc = reg.where(F.col("flow").isin("in", "book") & F.col("payee_s"))
risk = (inc.select("cadence", F.explode(F.sequence(F.trunc("first_dt", "month"),
                                                   F.trunc(F.coalesce("expected_dt", F.to_date(F.lit(W_END.isoformat()))), "month"),
                                                   F.expr("interval 1 month"))).alias("m"))
           .groupBy(F.date_format("m", "yyyy-MM").alias("month"), "cadence").count()
           .withColumnRenamed("count", "at_risk").toPandas())
ist = (inc.where("stopped").withColumn("month", F.date_format("expected_dt", "yyyy-MM")))
pw = ist.groupBy("payer", "month").agg(F.count("*").alias("n_rcv"))
ist = ist.join(pw, ["payer", "month"]).withColumn("payer_wide", F.col("n_rcv") >= MASS_STOP_MIN).cache()
sp = ist.groupBy("month", "cadence").agg(F.count("*").alias("stops"),
                                         F.sum(F.col("payer_wide").cast("long")).alias("pw")).toPandas()
mm = risk.merge(sp, on=["month", "cadence"], how="left").fillna({"stops": 0, "pw": 0})
mt = mm.groupby("month")[["at_risk", "stops", "pw"]].sum().reset_index()
t6d = pd.DataFrame({"month": mt.month, "stopped streams": mt.stops.astype("int64"),
                    "streams at risk": mt.at_risk.astype("int64"), "stop rate": mt.stops / mt.at_risk,
                    "payer-wide share": mt.pw / mt.stops.where(mt.stops > 0)})
for cd in ["weekly", "biweekly / semimonthly", "monthly"]:
    x = mm[mm.cadence == cd].set_index("month")
    t6d[f"rate {cd}"] = t6d.month.map(x.stops / x.at_risk)
t6d = t6d[t6d.month.isin(MONTHS)].sort_values("month")
PEAK = t6d.sort_values("stopped streams").month.iloc[-1] if len(t6d) else None

T_ENT_DIM = f"{DB}.pkg_cpty_entity_dim"
def _has(t, c):
    try:
        return c in spark.table(t).columns
    except Exception:
        return False
nm = []                                  # payer names: counterparty entities, then PNC customers (book payers)
if _has(T_ENT_DIM, "ent_name"):
    nm.append(spark.table(T_ENT_DIM).select(F.col("ent_id").alias("payer"), F.col("ent_name").alias("nm0")))
if _has(T_CUST, "customer_name"):
    nm.append(spark.table(T_CUST).select(F.col("mdm_id").alias("payer"), F.col("customer_name").alias("nm1")))
top = (ist.groupBy("payer", "payer_type").agg(
           F.sum((F.col("month") == PEAK).cast("long")).alias("stops in peak month"),
           F.count("*").alias("stops in window"),
           F.max(F.struct(F.col("n_rcv"), F.col("cadence"))).getField("cadence").alias("cadence"))
          .where(F.col("stops in peak month") > 0)
          .orderBy(F.desc("stops in peak month")).limit(15))
for n_ in nm:
    top = top.join(n_.dropDuplicates(["payer"]), "payer", "left")
top = top.toPandas()
top["payer_name"] = None
for c_ in ("nm0", "nm1"):
    if c_ in top:
        top["payer_name"] = top["payer_name"].fillna(top[c_])
top["other months, average"] = (top["stops in window"] - top["stops in peak month"]) / max(len(MONTHS) - 1, 1)
top = top.sort_values("stops in peak month", ascending=False)[
    ["payer_name", "payer", "payer_type", "cadence", "stops in peak month", "other months, average"]]
ist.unpersist()
out(sec("§6d Regular incoming streams: stops and stop rate by expected month", tbl(t6d, pct=tuple(t6d.columns[3:]))),
    sec(f"§6e Payers with the most stopped streams in {PEAK} (sampled customers)", tbl(top, dec=("other months, average",))),
    para(f"Payer-wide: the payer stopped at least {MASS_STOP_MIN} sampled customers in the month. Stop rate = stops / "
         "streams at risk in the month. Early months are low because a stream needs MIN_PAYMENTS payments first; the "
         "last months are low for long cadences."))

## §7 Customers who depend on regular income

Sampled customers that receive money (`in` + `book`, own transfers left out): share with any regular income, the share
of their incoming dollars that is regular, whether one regular stream brings at least half of their income, the
cadence of their largest regular income stream, and how many had a regular income stream stop in the window. This
sizes the population and the number of events for the contagion study.

In [ ]:
# ---- §7 Customers ------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

rc = ST.where(F.col("flow").isin("in", "book") & F.col("payee_s"))
cu7 = (rc.groupBy(F.col("payee").alias("cust"))
         .agg(F.sum("amt").alias("inc"), F.sum(F.when(F.col("regular"), F.col("amt")).otherwise(0)).alias("reg_amt"),
              F.sum(F.col("regular").cast("long")).alias("n_reg"),
              F.max(F.when(F.col("regular"), F.struct("amt", "cadence"))).alias("top"),
              F.max(F.col("stopped").cast("int")).alias("had_stop")))
seg = (spark.table(T_CUST).select(F.col("mdm_id").alias("cust"), F.upper(F.trim("party_type")).alias("pt"),
                                  F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("tm"))
            .dropDuplicates(["cust"]))
cu7 = (cu7.join(seg, "cust", "left")
          .withColumn("segment", F.concat(F.when(F.col("tm"), "TM ").otherwise("non-TM "),
                                          F.when(F.col("pt") == "O", "org").when(F.col("pt") == "P", "person")
                                           .otherwise("unknown")))
          .withColumn("reg_share", F.col("reg_amt") / F.col("inc"))
          .withColumn("top_share", F.col("top.amt") / F.col("inc"))
          .withColumn("top_cad", F.col("top.cadence")))
t7 = (cu7.groupBy("segment")
         .agg(F.count("*").alias("customers receiving money"),
              F.avg((F.col("n_reg") > 0).cast("double")).alias("any regular income"),
              F.percentile_approx("reg_share", 0.5, 1000).alias("median regular share of income"),
              F.avg((F.col("reg_share") >= 0.5).cast("double")).alias("regular ≥ 50% of income"),
              F.avg(F.coalesce(F.col("top_share") >= 0.5, F.lit(False)).cast("double")).alias("one regular stream ≥ 50%"),
              F.avg(F.col("had_stop").cast("double")).alias("had a regular income stream stop"))
         .toPandas().sort_values("customers receiving money", ascending=False))
tc = (cu7.where(F.col("n_reg") > 0).groupBy("segment", "top_cad").count().toPandas()
         .pivot_table(index="segment", columns="top_cad", values="count", aggfunc="sum", fill_value=0))
tc = tc.reindex(columns=[c for c in CAD_ORDER if c in tc.columns])
tc = tc.div(tc.sum(axis=1), axis=0).reset_index()
PCT7 = ("any regular income", "median regular share of income", "regular ≥ 50% of income", "one regular stream ≥ 50%",
        "had a regular income stream stop")
out(sec("§7a Receiving customers by segment", tbl(t7, pct=PCT7)),
    sec("§7b Cadence of the largest regular income stream (customers with regular income)",
        tbl(tc, pct=tuple(tc.columns[1:]))),
    para(f"Customers are the {SAMPLE_PCT}% sample; counts scale by about {100 / SAMPLE_PCT:.0f}× for all customers."))

## §8 Figures

Saves the main results as PNG files in `FIG_DIR/<window end>/` next to the notebook (nothing is drawn inline). The
cell lists the files it wrote.

In [ ]:
# ---- §8 Figures ---------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FDIR = os.path.join(FIG_DIR, END)
os.makedirs(FDIR, exist_ok=True)
SURF, INK, INK2, GRID, NEUTRAL = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2de", "#c3c2b7"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed order
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": NEUTRAL, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "legend.frameon": False, "lines.linewidth": 2})
CAD_COLOR = dict(zip(["daily", "weekly", "biweekly / semimonthly", "monthly", "bimonthly", "quarterly"],
                     [PAL[0], PAL[1], PAL[2], PAL[3], PAL[4], PAL[6]]))   # a cadence keeps its colour everywhere
FLOW_COLOR = {"in": PAL[0], "out": PAL[1], "book": PAL[2], "self": PAL[3]}
SAVED = []
def save(fig, name, what):
    fig.savefig(os.path.join(FDIR, name), dpi=160, bbox_inches="tight")
    plt.close(fig)
    SAVED.append({"file": name, "shows": what})
def stacked(ax, labels, parts, names, colors, min_label=0.08):
    """100% horizontal stacked bars, 2px surface gap between segments, labels on segments of at least min_label."""
    labels = [str(x) for x in labels]
    y = np.arange(len(labels)); left = np.zeros(len(labels))
    for vals, nm_, col in zip(parts, names, colors):
        v = np.nan_to_num(np.asarray(vals, dtype=float))
        ax.barh(y, v, left=left, color=col, edgecolor=SURF, linewidth=2, height=0.62, label=nm_)
        for i, (l0, w) in enumerate(zip(left, v)):
            if w >= min_label:
                ax.text(l0 + w / 2, i, f"{w:.0%}", ha="center", va="center", fontsize=7.5, color=INK)
        left += v
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="y", visible=False)
def bottom_legend(fig, ax, n):
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=min(n, 4), bbox_to_anchor=(0.5, -0.02))
tag = f"{MONTHS[0]} to {MONTHS[-1]}, {SAMPLE_PCT}% of customers"

# 1 regular share by flow
fl = [str(x) for x in t_flow.flow]
fig, axs = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
stacked(axs[0], fl, [t_flow[f"{g} (streams)"] for g in GROUPS], GROUPS, PAL[:4])
axs[0].set_title("Share of streams")
rest = 1 - t_flow["regular (dollars)"] - t_flow["recurring, irregular (dollars)"]
stacked(axs[1], fl, [t_flow["regular (dollars)"], t_flow["recurring, irregular (dollars)"], rest],
        ["regular", "recurring, irregular", "few payments or one-off"], [PAL[0], PAL[1], NEUTRAL])
axs[1].set_title("Share of dollars")
bottom_legend(fig, axs[0], 4); fig.suptitle(f"How much is regular, by flow ({tag})", x=0.01, ha="left", y=1.04)
fig.subplots_adjust(bottom=0.25); save(fig, "01_regular_share_by_flow.png", "regular / irregular / few / one-off, streams and dollars")

# 2 cadence mix of regular streams
cads = list(t_cad.cadence)
fig, axs = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
stacked(axs[0], FLOWS, [[t_cad.set_index("cadence").loc[c, f"{f} streams"] for f in FLOWS] for c in cads], cads,
        [CAD_COLOR.get(c, NEUTRAL) for c in cads])
axs[0].set_title("Regular streams")
stacked(axs[1], FLOWS, [[t_cad.set_index("cadence").loc[c, f"{f} dollars"] for f in FLOWS] for c in cads], cads,
        [CAD_COLOR.get(c, NEUTRAL) for c in cads])
axs[1].set_title("Regular dollars")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=len(cads), bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Cadence of regular streams, by flow", x=0.01, ha="left", y=1.04)
fig.subplots_adjust(bottom=0.25); save(fig, "02_cadence_mix.png", "cadence of regular streams and dollars per flow")

# 3 sensitivity to REG_MIN
row = sens[sens.MIN_PAYMENTS == MIN_PAYMENTS] if (sens.MIN_PAYMENTS == MIN_PAYMENTS).any() else sens.iloc[[1]]
xs = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
fig, ax = plt.subplots(figsize=(6, 3.2))
for i, (k, lab) in enumerate([("streams", "streams"), ("dollars", "dollars")]):
    ys = [float(row[f"REG_MIN {x} {k}"].iloc[0]) for x in xs]
    ax.plot(xs, ys, color=PAL[i], marker="o", markersize=5, label=lab)
    ax.text(xs[-1] + 0.01, ys[-1], lab, color=INK2, va="center", fontsize=8)
ax.axvline(REG_MIN, color=NEUTRAL, linestyle="--", linewidth=1); ax.text(REG_MIN, 1.02, f" REG_MIN = {REG_MIN}", fontsize=8, color=INK2)
ax.set_ylim(0, 1.05); ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_xlabel("share of gaps on time required")
ax.set_title(f"Regular share of all streams and dollars (MIN_PAYMENTS = {int(row.MIN_PAYMENTS.iloc[0])})"); ax.legend(loc="lower left")
save(fig, "03_sensitivity_reg_min.png", "regular share as REG_MIN changes")

# 4 amount spread around the stream's median
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, (_, r) in enumerate(d_cad.iterrows()):
    ys = [r[l] for l in DEV_LABELS]
    ax.plot(range(len(DEV_LABELS)), ys, color=CAD_COLOR.get(str(r.cadence), NEUTRAL), marker="o", markersize=4,
            label=str(r.cadence))
ax.set_xticks(range(len(DEV_LABELS))); ax.set_xticklabels(DEV_LABELS); ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_xlabel("within ± this much of the stream's median amount")
ax.set_ylabel("share of payments"); ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_title("How close regular payments are to their usual amount, by cadence")
save(fig, "04_amount_spread.png", "cumulative share of payments within ±x of the median amount")

# 5 amount class by cadence
fig, ax = plt.subplots(figsize=(8, 3.0))
stacked(ax, m_cad.cadence, [m_cad[f"{x} (streams)"] for x in AC], AC, [PAL[6], PAL[2], PAL[3], NEUTRAL])
h, l = ax.get_legend_handles_labels(); ax.legend(h, l, loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.12))
ax.set_title("Amount class of regular streams, by cadence (share of streams)")
save(fig, "05_amount_class.png", "fixed / stepped / bounded / variable by cadence")

# 6 days after the usual gap
labs = [l for l, _, _ in LAG_BINS]; rows_ = list(t5b.iterrows())
fig, axs = plt.subplots(1, len(rows_), figsize=(2.1 * len(rows_), 2.8), sharey=True)
axs = np.atleast_1d(axs)
for ax, (_, r) in zip(axs, rows_):
    ax.bar(range(len(labs)), [r[l] for l in labs], color=PAL[0], width=0.75)
    ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=60, fontsize=7)
    ax.set_title(str(r.cadence).replace(" / ", " /\n"), fontsize=9)
    ax.grid(axis="x", visible=False)
axs[0].yaxis.set_major_formatter(PercentFormatter(1.0)); axs[0].set_ylabel("share of gaps")
fig.suptitle("Days after the usual gap (negative = early)", x=0.01, ha="left", y=1.05)
save(fig, "06_lateness.png", "distribution of days early / late by cadence")

# 7 stop rate by cadence and flow
bb = b[b.flow.astype(str).isin(["in", "out", "book"])]
cads7 = [c for c in CAD_ORDER if c in set(bb.cadence.astype(str))]
fig, ax = plt.subplots(figsize=(8, 3.2)); w = 0.26
for i, f in enumerate(["in", "out", "book"]):
    x = bb[bb.flow.astype(str) == f].set_index(bb[bb.flow.astype(str) == f].cadence.astype(str))
    ax.bar(np.arange(len(cads7)) + (i - 1) * w, [x["stops per 100 stream-months"].get(c, np.nan) for c in cads7],
           width=w, color=FLOW_COLOR[f], edgecolor=SURF, linewidth=1, label=f)
ax.set_xticks(range(len(cads7))); ax.set_xticklabels(cads7); ax.grid(axis="x", visible=False)
ax.set_ylabel("stops per 100 stream-months"); ax.legend(); ax.set_title("How often regular streams stop, by cadence and flow")
save(fig, "07_stop_rate.png", "stops per 100 stream-months by cadence and flow")

# 8 stops month by month
if len(t6d):
    fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    pw_n = (t6d["payer-wide share"].fillna(0) * t6d["stopped streams"]).values
    x = np.arange(len(t6d))
    axs[0].bar(x, t6d["stopped streams"] - pw_n, color=PAL[0], edgecolor=SURF, linewidth=1, label="other stops")
    axs[0].bar(x, pw_n, bottom=t6d["stopped streams"] - pw_n, color=PAL[1], edgecolor=SURF, linewidth=1, label="payer-wide")
    axs[0].set_title("Stopped regular incoming streams by expected month"); axs[0].legend(); axs[0].grid(axis="x", visible=False)
    for i, cd in enumerate(["weekly", "biweekly / semimonthly", "monthly"]):
        axs[1].plot(x, t6d[f"rate {cd}"], color=CAD_COLOR[cd], marker="o", markersize=4, label=cd)
    axs[1].yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1)); axs[1].set_title("Stop rate (stops / streams at risk)")
    axs[1].legend(); axs[1].set_xticks(x); axs[1].set_xticklabels(list(t6d.month), rotation=45)
    save(fig, "08_stops_by_month.png", "incoming stops per month (payer-wide part) and stop rate by cadence")

# 9 customer dependency on regular income
mets = ["any regular income", "regular ≥ 50% of income", "one regular stream ≥ 50%", "had a regular income stream stop"]
t7p = t7[t7["customers receiving money"] >= 1000] if (t7["customers receiving money"] >= 1000).any() else t7
fig, ax = plt.subplots(figsize=(8, 3.2)); w = 0.2; segs = list(t7p.segment)
for i, m_ in enumerate(mets):
    ax.bar(np.arange(len(segs)) + (i - 1.5) * w, t7p[m_], width=w, color=PAL[i], edgecolor=SURF, linewidth=1, label=m_)
ax.set_xticks(range(len(segs))); ax.set_xticklabels(segs); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="x", visible=False); ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8)
ax.set_title("Receiving customers: dependence on regular income (segments with ≥ 1,000 sampled customers)")
save(fig, "09_customer_dependency.png", "regular income share, dependence and stops by segment")

# 10 cadence of the largest regular income stream
tcs = tc[tc.segment.isin(segs)] if len(tc) else tc
if len(tcs):
    cc = list(tcs.columns[1:])
    fig, ax = plt.subplots(figsize=(8, 2.6))
    stacked(ax, tcs.segment, [tcs[c] for c in cc], cc, [CAD_COLOR.get(c, NEUTRAL) for c in cc])
    h, l = ax.get_legend_handles_labels(); ax.legend(h, l, loc="upper center", ncol=len(cc), bbox_to_anchor=(0.5, -0.15), fontsize=8)
    ax.set_title("Cadence of the customer's largest regular income stream")
    save(fig, "10_income_cadence.png", "cadence of the largest regular income stream by segment")

out(sec(f"§8 Figures: {len(SAVED)} PNG files in {os.path.abspath(FDIR)}", tbl(pd.DataFrame(SAVED))))

## §9 Housekeeping

In [ ]:
# ---- §9 Housekeeping ----------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"
others = sorted(set(ls_dirs(SCR)) - {f"sd_{SIG_SD}", f"st_{SIG_ST}"})
sizes = pd.DataFrame([{"folder": o, "GB": size_gb(f"{SCR}/{o}")} for o in others])
if CLEAN_OTHER_SCRATCH:
    for o in others:
        delete(f"{SCR}/{o}")
cur = {"stream days": size_gb(P_SD), "streams": size_gb(P_ST)}
ST.unpersist()
if not KEEP_SCRATCH:
    delete(P_SD); delete(P_ST)
times = pd.DataFrame([{"step": k, "minutes": v} for k, v in TIMES.items()])
out(sec("§9 Scratch", tbl(pd.DataFrame([{"scratch": k, "GB": v} for k, v in cur.items()]), dec=("GB",))),
    sec("Other settings' scratch" + (" (deleted)" if CLEAN_OTHER_SCRATCH else ""), tbl(sizes, dec=("GB",)))
    if len(sizes) else "",
    sec("Build time", tbl(times, dec=("minutes",))) if len(times) else "",
    para("Scratch kept for reruns of §3–§7." if KEEP_SCRATCH else "Scratch deleted."))